# 07. g3 통합 학습 — 바뀐 것만 전부 재튜닝하고 확정 모델과 비교

## 원칙 — 먼저 못 박고 시작한다

**모델이나 학습 데이터를 바꾸면 그 구조에서 하이퍼파라미터를 처음부터 다시 튜닝하고, 그 최종 결과를 확정 모델 성능과 비교한다.**

옛 설정을 새 구조에 씌우면 **새 구조에 불리한 조건을 걸어놓고 재는 것**이라 결과가 무의미하다.

> 2026-07-24에 이미 겪은 교훈 — "손실을 바꿨으면 `T_soft`·블렌드도 다시 튜닝해야 공정하다"는 지적으로 재비교하니 **악화로 보이던 것이 개선으로 뒤집혔다.**

**그리고 뒤집어서, 안 바뀐 것은 건드리지 않는다.** g1·g2는 데이터도 구조도 그대로다. 거기까지 다시 튜닝하면 비용만 세 배가 되고, `06`에서 리더보드 역행이 확인된 "눈금 재적합"을 g1·g2에 또 하는 셈이 된다.

| | g1 · g2 | **g3** |
|---|---|---|
| 학습 데이터 | **안 바뀜** | **4,847 → 26,686행 (5.5배)** |
| 튜닝 | **exp017 설정 유지** | **처음부터 전부 재튜닝** |

## 무엇을 재튜닝하는가 — 전수

| 대상 | 파라미터 | 왜 |
|---|---|---|
| **LightGBM** | `alpha`(τ)·`learning_rate`·`num_leaves`·**`min_data_in_leaf`**·`feature_fraction`·`bagging_fraction`·`lambda_l1/l2` | 학습 행이 5배면 **잎당 최소 샘플**부터 달라져야 한다. 트리 수는 early stopping 자동 |
| **MLP 구조** | **은닉층 9가지 전수** — (256,256)/(128,128)/(384,384)/(512,512)/(256,256,256)/(384,384,384)/**(512,256)**/(384,192)/(512,256,128) | 데이터가 많아지면 더 넓고 깊은 모델을 감당한다. **좁아지는 구조도 넣었다** |
| **MLP 최적화** | **`lr`(격자 전수 후 미세조정)·`weight_decay`·`p_drop`** | ★ 이 MLP는 **full-batch 학습**이다(`metric_loss`의 FICR 항이 '전체 합의 비율'이라 미니배치면 추정이 흔들린다). **배치 크기 = 데이터 크기**이므로 표본이 5.5배가 되면 **기울기의 성격 자체가 달라진다.** 학습률을 그대로 두면 안 된다 |
| **손실** | `T_soft` | 계단 근사 폭 |
| **블렌드** | `w` (g3) | 섞을 두 모델 중 하나라도 바뀌면 비율이 바뀐다 |

이를 위해 `src/nn.py`의 `train_mlp()`/`MLP`가 구조·최적화 인자를 받도록 열었다. **기본값이 예전 상수와 같아 `state_dict` 키까지 하위 호환**이며, 저장된 exp017 가중치가 그대로 읽히는 것과 기본 인자 예측이 비트 단위로 동일한 것을 확인했다.

## 비교 설계 — 세 칸

| 칸 | g3 LightGBM | g3 MLP | 정체 |
|---|---|---|---|
| **A** | 전용(exp017) | 전용(exp017) | **확정 모델** exp017, Public 0.638863 |
| **B** | 전용(exp017) | **통합 + 전체 재튜닝** | MLP만 통합 |
| **C** | **통합 + 재튜닝** | **통합 + 전체 재튜닝** | 둘 다 통합 |

| 비교 | 묻는 것 |
|---|---|
| **B − A**, **C − A** | **확정 모델을 넘었는가** ← 제출 판단 |
| **C − B** | 통합이 **LightGBM에도** 듣는가 |

`06` §5가 잰 A′(완전분리 + 재튜닝) = **0.6487**은 같은 하네스·같은 시드의 결정론적 결과라 필요하면 그 숫자를 인용한다. 다시 돌리지 않는다.

## 예비 신호 (작성 중 실측)

같은 설정으로 붙여 본 g3 내부검증 점수다.

| | 학습 행 | g3 내부검증 |
|---|---:|---:|
| 전용 · 256폭 2층 | 4,847 | 0.5707 |
| 통합 · 256폭 2층 | 19,060 | **0.5914** |
| 통합 · **256폭 3층** | 19,060 | **0.5954** |

**통합만으로 +0.021, 층을 늘리니 +0.025.** 구조 튜닝이 실제로 값을 한다는 뜻이라 이 노트북의 투자가 정당화된다. (LightGBM도 같은 파라미터로 통합하니 0.5544 → 0.5670이었다.)

## 판정

홀드아웃 이득은 `06` §13에서 리더보드 이득을 예측하지 못하는 것이 확인됐다. **(a) 코드가 망가지지 않았는지 확인**하고 **(b) 제출할 가치가 있는지 정하는** 데만 쓴다. **채택은 리더보드가 한다.**

> ⚠️ curtailment 행 제외는 이 노트북에서 뺐다. 성격이 다른 별개 레버라 섞으면 무엇이 일했는지 흐려진다. HANDOFF 권장 순서 4번으로 남겨 둔다.

## 0. 준비

In [7]:
import os, random, json, sys, time, subprocess, platform, warnings
from pathlib import Path
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다."

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import torch

sys.path.insert(0, str(REPO_ROOT))
from src.metric import metric, TARGET_COLS, CAPACITY_KWH
from src.nn import (T_SOFT, HIDDEN, DROPOUT, LR, WEIGHT_DECAY, MAX_EPOCHS,
                    set_seed, fit_standardizer, standardize, train_mlp, predict_mlp)

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

RNG_SEED = 42
set_seed(RNG_SEED)
N_THREADS = 4
torch.set_num_threads(N_THREADS)

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
EXP_DIR = REPO_ROOT / "experiments"; EXP_DIR.mkdir(exist_ok=True)

SCORE_THRESHOLD = 0.10
NN_SEEDS = [42, 1337, 2024, 7, 99]     # 최종 학습·채점용
SEARCH_SEEDS = [42, 1337]              # 탐색용 (통합 MLP는 시드당 25~35초라 5개는 비싸다)
MAX_ROUNDS, EARLY_STOP = 2000, 50
G3 = "kpx_group_3"; GI3 = TARGET_COLS.index(G3)

# ---------- exp017 설정. g1·g2는 이 값을 그대로 쓰고, 비교 기준이므로 바꾸지 않는다 ----------
EXP017_TAU = {"kpx_group_1": 0.70, "kpx_group_2": 0.50, "kpx_group_3": 0.65}
EXP017_W   = {"kpx_group_1": 0.8,  "kpx_group_2": 0.5,  "kpx_group_3": 0.9}
EXP017_LGB_CORE = dict(learning_rate=0.05, num_leaves=63, min_data_in_leaf=40,
                   feature_fraction=0.7, bagging_fraction=0.8, lambda_l2=1.0)
LGB_BASE = dict(objective="quantile", bagging_freq=1, verbosity=-1, seed=RNG_SEED,
                num_threads=N_THREADS, deterministic=True, force_row_wise=True)
EXP017_MLP = dict(hidden=HIDDEN, p_drop=DROPOUT, n_layers=2, lr=LR,
              weight_decay=WEIGHT_DECAY, t_soft=T_SOFT)

VAL_START       = pd.Timestamp("2024-01-01 01:00:00")   # 2024는 채점 전용
INNER_VAL_START = pd.Timestamp("2023-07-01 01:00:00")   # 튜닝은 전부 이 안에서

TAU_GRID = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
W_GRID   = np.round(np.arange(0.0, 1.001, 0.05), 2)
N_TRIALS_LGB, N_TRIALS_MLP = 15, 15

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print(f"Python {platform.python_version()} | LightGBM {lgb.__version__} | PyTorch {torch.__version__}")
print(f"exp017 설정: τ {EXP017_TAU} | w {EXP017_W}")
print(f"exp017 MLP : {EXP017_MLP}")
print(f"탐색 예산  : LightGBM τ격자{len(TAU_GRID)}+Optuna{N_TRIALS_LGB} | MLP Optuna{N_TRIALS_MLP}"
      f" (탐색 시드 {SEARCH_SEEDS}, 최종 시드 {NN_SEEDS})")

Python 3.13.14 | LightGBM 4.6.0 | PyTorch 2.13.0+cpu
챔피언 설정: τ {'kpx_group_1': 0.7, 'kpx_group_2': 0.5, 'kpx_group_3': 0.65} | w {'kpx_group_1': 0.8, 'kpx_group_2': 0.5, 'kpx_group_3': 0.9}
챔피언 MLP : {'hidden': 256, 'p_drop': 0.15, 'n_layers': 2, 'lr': 0.001, 'weight_decay': 0.0001, 't_soft': 0.006}
탐색 예산  : LightGBM τ격자6+Optuna15 | MLP Optuna15 (탐색 시드 [42, 1337], 최종 시드 [42, 1337, 2024, 7, 99])


**실행 후 확인할 것**

- 커널이 `wind_forecast (venv)` 인지.
- exp017 MLP가 `hidden=256, p_drop=0.15, n_layers=2, lr=0.001, weight_decay=0.0001, t_soft=0.006` 인지.

## 1. 피처 로딩과 홀드아웃 정의

In [8]:
feat = pd.read_parquet(PROCESSED_DIR / "features_train.parquet")
FEATURE_COLS = [c for c in feat.columns if c not in TARGET_COLS + ["forecast_kst_dtm"]]
assert feat[FEATURE_COLS].isna().sum().sum() == 0 and not np.isinf(feat[FEATURE_COLS].to_numpy()).any()

dtm = feat["forecast_kst_dtm"]
holdout_mask = dtm >= VAL_START
HOLDOUT_ANS = feat.loc[holdout_mask, TARGET_COLS].reset_index(drop=True)
X_ALL = feat[FEATURE_COLS].to_numpy(np.float32)      # 반복 변환을 피하려고 한 번만


def build_masks(g):
    lab = feat[g].notna(); sc = feat[g] >= CAPACITY_KWH[g] * SCORE_THRESHOLD
    bv, bi = dtm < VAL_START, dtm < INNER_VAL_START
    return {"fit_all": lab & bv, "fit_scored": lab & sc & bv,
            "inner_tr": lab & bi, "inner_tr_scored": lab & sc & bi,
            "inner_va": lab & (~bi) & bv, "inner_va_scored": lab & sc & (~bi) & bv}


MASKS = {g: build_masks(g) for g in TARGET_COLS}
print(f"피처 {len(FEATURE_COLS)}개 | 홀드아웃 {int(holdout_mask.sum()):,}행 (2024)\n")
for g in TARGET_COLS:
    m = MASKS[g]
    print(f"  {g}: 최종학습 {int(m['fit_all'].sum()):,} (채점 {int(m['fit_scored'].sum()):,}) "
          f"| 내부학습(채점) {int(m['inner_tr_scored'].sum()):,} | 내부검증 {int(m['inner_va'].sum()):,}")

피처 179개 | 홀드아웃 8,784행 (2024)

  kpx_group_1: 최종학습 17,422 (채점 10,925) | 내부학습(채점) 8,273 | 내부검증 4,413
  kpx_group_2: 최종학습 17,423 (채점 10,914) | 내부학습(채점) 8,251 | 내부검증 4,414
  kpx_group_3: 최종학습 8,760 (채점 4,847) | 내부학습(채점) 2,536 | 내부검증 4,416


**실행 후 확인할 것**

- `kpx_group_3`의 최종학습이 다른 그룹의 **정확히 절반**(8,760 vs 17,422)이어야 한다. 이 절이 이 노트북의 출발점이다.

## 2. 산식과 통합 데이터 구성

### 통합할 때 타깃을 이용률로 바꾸는 이유

설비용량이 다르다(21,600 / 21,600 / 21,000). kWh를 그대로 쌓으면 **같은 "잘한 정도"가 그룹마다 다른 숫자**가 된다. 그래서 타깃·가중치를 **이용률**(발전량 ÷ 설비용량)로 통일하고, 예측한 뒤 설비용량을 곱해 되돌린다. MLP가 이미 쓰는 방식이라 일관되기도 하다.

그룹 구분은 **원-핫 3개**로 알려 준다 (179 + 3 = 182열).

In [9]:
def group_score(actual, forecast, capacity):
    """metric()과 같은 산식을 한 그룹만 계산하도록 뗀 것.
    src/metric.py는 대회 공식 산식이라 수정 금지(CLAUDE.md 4번)여서 재정의한다."""
    a = np.asarray(actual, float); f = np.asarray(forecast, float)
    v = a >= capacity * 0.10
    if not np.any(v):
        return float("nan"), float("nan"), float("nan")
    a, f = a[v], f[v]
    er = np.abs(f - a) / capacity
    nmae = er.mean()
    price = np.select([er <= 0.06, er <= 0.08], [4.0, 3.0], default=0.0)
    return 0.5 * (1 - nmae) + 0.5 * (a * price).sum() / (a * 4.0).sum(), nmae, None


def onehot(X, gi):
    o = np.zeros((len(X), len(TARGET_COLS)), np.float32); o[:, gi] = 1.0
    return np.hstack([X, o])


def stack_groups(mask_key):
    """3그룹을 세로로 쌓는다. 타깃은 이용률."""
    Xs, ys = [], []
    for gi, g in enumerate(TARGET_COLS):
        m = MASKS[g][mask_key].to_numpy()
        Xs.append(onehot(X_ALL[m], gi))
        ys.append((feat.loc[m, g].to_numpy(np.float64)) / CAPACITY_KWH[g])
    return np.vstack(Xs), np.concatenate(ys)


X_u_inner, y_u_inner = stack_groups("inner_tr_scored")
X_u_full,  y_u_full  = stack_groups("fit_scored")
print(f"통합 MLP 학습셋 — 내부 {X_u_inner.shape} | 최종 {X_u_full.shape}")
print(f"  g3 전용 대비: 내부 {len(y_u_inner)/int(MASKS[G3]['inner_tr_scored'].sum()):.1f}배 "
      f"| 최종 {len(y_u_full)/int(MASKS[G3]['fit_scored'].sum()):.1f}배")
print(f"  타깃 범위 [{y_u_full.min():.3f}, {y_u_full.max():.3f}] (이용률)")

통합 MLP 학습셋 — 내부 (19060, 182) | 최종 (26686, 182)
  g3 전용 대비: 내부 7.5배 | 최종 5.5배
  타깃 범위 [0.100, 1.004] (이용률)


**실행 후 확인할 것**

- 통합 최종 학습셋이 **(26,686 × 182)**, g3 전용 대비 **5.5배**여야 한다.
- 타깃이 0~1 근방(이용률)이어야 한다.

## 3. LightGBM — 전용 / 통합

In [10]:
def train_lgb_group(g, params):
    m = MASKS[g]; cap = CAPACITY_KWH[g]
    d_tr = lgb.Dataset(feat.loc[m["inner_tr"], FEATURE_COLS], label=feat.loc[m["inner_tr"], g],
                       weight=feat.loc[m["inner_tr"], g].to_numpy())
    d_va = lgb.Dataset(feat.loc[m["inner_va_scored"], FEATURE_COLS],
                       label=feat.loc[m["inner_va_scored"], g], reference=d_tr)
    probe = lgb.train(params, d_tr, MAX_ROUNDS, valid_sets=[d_va],
                      callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    inner = np.clip(probe.predict(feat.loc[m["inner_va"], FEATURE_COLS],
                                  num_iteration=probe.best_iteration), 0, cap)
    n = max(int(round(probe.best_iteration * m["fit_all"].sum() / m["inner_tr"].sum())), 50)
    b = lgb.train(params, lgb.Dataset(feat.loc[m["fit_all"], FEATURE_COLS],
                                      label=feat.loc[m["fit_all"], g],
                                      weight=feat.loc[m["fit_all"], g].to_numpy()), n)
    ho = np.clip(b.predict(feat.loc[holdout_mask, FEATURE_COLS]), 0, cap)
    return ho, inner, {"mode": "per-group", "n_trees": int(b.num_trees())}


def train_lgb_unified_g3(params):
    """3그룹을 합쳐 학습하고 g3를 예측한다. 타깃·가중치는 이용률."""
    m = MASKS[G3]; cap = CAPACITY_KWH[G3]
    Xtr, ytr = stack_groups("inner_tr")
    d_tr = lgb.Dataset(Xtr, label=ytr, weight=ytr)
    d_va = lgb.Dataset(onehot(X_ALL[m["inner_va_scored"].to_numpy()], GI3),
                       label=(feat.loc[m["inner_va_scored"], G3] / cap).to_numpy(), reference=d_tr)
    probe = lgb.train(params, d_tr, MAX_ROUNDS, valid_sets=[d_va],
                      callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    inner = np.clip(probe.predict(onehot(X_ALL[m["inner_va"].to_numpy()], GI3),
                                  num_iteration=probe.best_iteration), 0, 1) * cap
    Xf, yf = stack_groups("fit_all")
    n = max(int(round(probe.best_iteration * len(yf) / len(ytr))), 50)
    b = lgb.train(params, lgb.Dataset(Xf, label=yf, weight=yf), n)
    ho = np.clip(b.predict(onehot(X_ALL[holdout_mask.to_numpy()], GI3)), 0, 1) * cap
    return ho, inner, {"mode": "unified", "n_trees": int(b.num_trees())}


def g3_inner_score(pred):
    """g3 내부검증(2023 하반기) 대회 점수. 튜닝의 목적함수."""
    return group_score(feat.loc[MASKS[G3]["inner_va"], G3].to_numpy(), pred, CAPACITY_KWH[G3])[0]


print("LightGBM 두 갈래 준비 완료")

LightGBM 두 갈래 준비 완료


**실행 후 확인할 것**

- 함수 정의만 하는 셀이다.

## 4. MLP — 전용 / 통합, 구조를 인자로 받는다

`src/nn.py`를 고쳐 `hidden`·`n_layers`·`p_drop`·`lr`·`weight_decay`를 인자로 넘길 수 있게 했다. 기본값이 예전 상수와 같아 **하위 호환**이다(저장된 exp017 가중치가 그대로 읽히고, 기본 인자 예측이 비트 단위로 동일한 것을 확인).

**조기 종료는 g3 내부검증 점수로 판단한다** — 통합 모델이지만 우리가 올리려는 건 g3 성적이다.

In [11]:
def train_mlp_cfg(g, cfg, seeds, unified, need_full=True):
    """MLP 한 벌 학습. cfg = hidden/p_drop/n_layers/lr/weight_decay/t_soft.
    돌려주는 것: (내부검증 예측 평균, 에폭, 홀드아웃 평균, 홀드아웃 시드별, 학습행수)"""
    cap = CAPACITY_KWH[g]; m = MASKS[g]
    # hidden은 정수(모든 층 같은 폭) 또는 튜플(층별 폭)이다. src/nn.py의 MLP가 둘 다 받는다.
    arch = {k: cfg[k] for k in ("hidden", "p_drop", "n_layers", "lr", "weight_decay")}
    ts = cfg["t_soft"]

    if unified:
        X_itr, y_itr = X_u_inner, y_u_inner.astype(np.float32)
    else:
        X_itr = X_ALL[m["inner_tr_scored"].to_numpy()]
        y_itr = (feat.loc[m["inner_tr_scored"], g] / cap).to_numpy(np.float32)
    X_ivs = X_ALL[m["inner_va_scored"].to_numpy()]
    y_ivs = (feat.loc[m["inner_va_scored"], g] / cap).to_numpy(np.float32)
    X_iva = X_ALL[m["inner_va"].to_numpy()]
    if unified:
        X_ivs, X_iva = onehot(X_ivs, GI3), onehot(X_iva, GI3)

    mu, sd = fit_standardizer(X_itr)
    Xi, Xv = (X_itr - mu) / sd, standardize(X_ivs, mu, sd)

    def eval_fn(model):
        with torch.no_grad():
            p = np.clip(model(Xv).numpy(), 0, 1)
        return group_score(y_ivs * cap, p * cap, cap)[0]

    inner, epochs = [], []
    for s in seeds:
        mdl, be = train_mlp(Xi, y_itr, seed=s, n_epochs=MAX_EPOCHS, t_soft=ts, eval_fn=eval_fn, **arch)
        epochs.append(be); inner.append(predict_mlp(mdl, X_iva, mu, sd, cap))
    ep = int(np.median(epochs))
    if not need_full:
        return np.mean(inner, axis=0), ep, None, None, len(y_itr)

    if unified:
        X_f, y_f = X_u_full, y_u_full.astype(np.float32)
    else:
        X_f = X_ALL[m["fit_scored"].to_numpy()]
        y_f = (feat.loc[m["fit_scored"], g] / cap).to_numpy(np.float32)
    mu2, sd2 = fit_standardizer(X_f)
    Xf = (X_f - mu2) / sd2
    X_ho = X_ALL[holdout_mask.to_numpy()]
    if unified:
        X_ho = onehot(X_ho, GI3)
    ho = np.array([predict_mlp(train_mlp(Xf, y_f, seed=s, n_epochs=ep, t_soft=ts, eval_fn=None, **arch)[0],
                               X_ho, mu2, sd2, cap) for s in NN_SEEDS])
    return np.mean(inner, axis=0), ep, ho.mean(axis=0), ho, len(y_f)


print("MLP 준비 완료")

MLP 준비 완료


**실행 후 확인할 것**

- 함수 정의만 하는 셀이다.

## 5. g3 재튜닝 — 데이터가 바뀐 것만

### 5-1. 통합 LightGBM (τ 격자 + Optuna)

**학습 데이터가 실제로 바뀌었으므로 반드시 다시 뽑는다.** 목적함수는 **g3 내부검증 점수**(2023 하반기)다. 2024는 보지 않는다.

⏱️ **약 8~10분**

In [ ]:
def lgb_obj_from(params):
    _, inner, _ = train_lgb_unified_g3(params)
    return g3_inner_score(inner)


t0 = time.time()
print("=== 5-1. g3 통합 LightGBM 튜닝 ===")
grid = [(lgb_obj_from({**LGB_BASE, **EXP017_LGB_CORE, "alpha": t}), t) for t in TAU_GRID]
cv_grid, best_tau = max(grid)
print(f"  τ 격자 최고 {cv_grid:.4f} (τ={best_tau})")


def lgb_objective(tr):
    return lgb_obj_from({**LGB_BASE,
        "alpha": tr.suggest_float("alpha", 0.45, 0.80),
        "learning_rate": tr.suggest_float("learning_rate", 0.02, 0.10, log=True),
        "num_leaves": tr.suggest_int("num_leaves", 15, 255, log=True),
        "min_data_in_leaf": tr.suggest_int("min_data_in_leaf", 20, 400, log=True),
        "feature_fraction": tr.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": tr.suggest_float("bagging_fraction", 0.5, 1.0),
        "lambda_l1": tr.suggest_float("lambda_l1", 1e-3, 10.0, log=True),
        "lambda_l2": tr.suggest_float("lambda_l2", 1e-3, 10.0, log=True)})


st = optuna.create_study(direction="maximize",
                        sampler=optuna.samplers.TPESampler(seed=RNG_SEED, n_startup_trials=5))
st.enqueue_trial({**EXP017_LGB_CORE, "alpha": best_tau, "lambda_l1": 1e-3})   # 알려진 좋은 점에서 출발
st.optimize(lgb_objective, n_trials=N_TRIALS_LGB, show_progress_bar=False)
print(f"  Optuna 최고 {st.best_value:.4f}")

if st.best_value > cv_grid:
    LGB_UNI_G3 = {**LGB_BASE, **st.best_params}
else:
    LGB_UNI_G3 = {**LGB_BASE, **EXP017_LGB_CORE, "alpha": best_tau}
print(f"  -> 채택 alpha={LGB_UNI_G3['alpha']:.4f} lr={LGB_UNI_G3['learning_rate']:.4f} "
      f"leaves={LGB_UNI_G3['num_leaves']} min_leaf={LGB_UNI_G3['min_data_in_leaf']} "
      f"| 소요 {time.time()-t0:.0f}초")
print(f"  [대조] exp017 g3 전용: alpha={EXP017_TAU[G3]} lr=0.05 leaves=63 min_leaf=40")

=== 5-1. g3 통합 LightGBM 튜닝 ===


**실행 후 확인할 것**

- **`min_data_in_leaf`가 40(확정 모델)보다 크게 나오는지** 본다. 학습 행이 5배가 됐으니 커지는 게 자연스럽고, 그러면 "데이터가 바뀌면 눈금도 바뀐다"가 실제로 확인된 것이다.
- `enqueue_trial`로 exp017 코어값에서 출발하므로 Optuna 최고가 τ 격자보다 못할 일은 거의 없다.

### 5-2. 통합 MLP 튜닝 — 3단계 순차

### 왜 한꺼번에 안 하고 나누는가

교과서적 순서는 **학습률이 1순위**다(구조가 아니다). 학습률이 어긋나면 **구조를 아무리 잘 골라도 그 효과가 안 보인다** — 학습 자체가 제대로 안 되기 때문이다.

그런데 순차 튜닝에는 구멍이 있다. **lr과 구조는 서로 얽혀 있어서**, 넓고 깊은 망은 보통 더 작은 lr을 원한다. `(256,256)`에서 고른 lr을 `(512,512)`에 씌우면 **그 구조에 불리한 조건을 걸어놓고 재는 셈**이다. 그래서 요즘은 순차 대신 동시 탐색(Optuna)을 쓴다.

**우리는 예산이 작아서 동시 탐색이 얇다** — 15회로 6개 축을 훑는데 구조만 9가지라 구조당 평균 1.7회다. 각 구조에 맞는 lr을 찾아줄 여유가 없다.

**그래서 순차로 하되, 3단계에서 lr을 다시 맞춰 순차의 구멍을 메운다.**

| 단계 | 고정 | 탐색 | 방식 | 회차 |
|---|---|---|---|---|
| **1** | 구조 `(256,256)` 등 exp017 값 전부 | **`lr`** | 격자 전수 | 7 |
| **2** | 1단계 최적 `lr` + 나머지 exp017 값 | **은닉층 구조 9가지** | 전수 | 9 |
| **3** | 2단계 최적 구조 | **`lr`·`p_drop`·`weight_decay`·`t_soft`** | Optuna | 8 |

### 1단계 — `lr` 격자

| | |
|---|---|
| 후보 | `2e-4, 3e-4, 5e-4, 1e-3, 2e-3, 3e-3, 5e-3` (1e-3 = exp017 값) |
| 범위 근거 | **배치가 커지면 학습률도 커지는 쪽이 맞는 경우가 많다.** 이 MLP는 full-batch라 **배치 크기 = 데이터 크기**인데 4,847 → 26,686행으로 5.5배가 됐다. 그래서 exp017 값(1e-3) **위쪽까지** 열었다 |

⚠️ `GRAD_CLIP=1.0`이 걸려 있어 기울기가 계속 잘리면 실효 학습률이 또 달라진다. 격자 경계에서 최적이 나오면 그 방향으로 더 열어야 한다(셀이 경고한다).

### 2단계 — 은닉층 구조 9가지 전수

범위(`128~512`)로 두지 않고 **목록으로 명시**한다. 범위로 두면 `[293, 293]` 같은 값이 뽑혀 예산만 흩어지고, 무엇보다 **`(512, 256)`처럼 좁아지는 구조를 표현할 수 없다**(모든 층이 같은 폭이 되므로).

| # | 은닉층 | 성격 |
|---|---|---|
| 0 | **(256, 256)** | **exp017 구조** — 비교 기준 |
| 1 | (128, 128) | 더 작게 |
| 2 | (384, 384) | 더 넓게 |
| 3 | (512, 512) | 훨씬 넓게 |
| 4 | (256, 256, 256) | 더 깊게 |
| 5 | (384, 384, 384) | 넓고 깊게 |
| 6 | (512, 256) | **좁아지는 형태** |
| 7 | (384, 192) | 〃 |
| 8 | (512, 256, 128) | 〃 3층 |

### 3단계 — 미세조정 (여기서 lr을 다시 본다)

2단계에서 고른 구조 위에서 `lr`·`p_drop`·`weight_decay`·`t_soft`를 **Optuna로 동시에** 다시 훑는다. 1단계의 lr은 `(256,256)` 기준이었으므로 **구조가 바뀌었으면 최적 lr도 움직인다** — 이 단계가 그걸 잡는다. 1단계 최적값을 첫 시행으로 넣어 거기서 출발한다.

> **`n_startup_trials=5`**: Optuna의 TPE는 초기 몇 회를 무작위로만 뽑고 그 뒤부터 모델을 쓴다. 기본값 10이면 15회 중 **5회만** 모델 안내를 받는다. 첫 시행에 exp017 값을 넣어 좋은 점을 하나 알고 시작하므로 5로 낮춰 **10회가 안내를 받게** 했다. 시간은 그대로다.

| 파라미터 | 범위 |
|---|---|
| `lr` | 1단계 최적의 **1/3배 ~ 3배** |
| `p_drop` | 0.0 ~ 0.30 |
| `weight_decay` | 1e-5 ~ 1e-3 (로그) |
| `t_soft` | 0.003 / 0.006 / 0.012 |

### 이번에 건드리지 않는 것 (고정)

| 고정 | 값 |
|---|---|
| 최적화기 / 스케줄 | AdamW / 코사인 |
| 활성함수 / 정규화 | GELU / BatchNorm |
| 기울기 클리핑 | 1.0 |
| dropout | **전 층 동일** |
| 조기 종료 | `PATIENCE=60`, `EVAL_EVERY=5`, 최대 400에폭 |
| 통합 시 그룹 가중 | **1 : 1 : 1** |

**탐색은 시드 2개**로 한다(통합 MLP는 시드당 25~35초라 5개는 비싸다). 최종 학습·채점만 시드 5개를 쓴다.

⏱️ **총 24회 ≈ 25분**

In [ ]:
t0 = time.time()
print("=== 5-2. g3 통합 MLP 튜닝 (3단계) ===")

# 출발점 — exp017 설정을 전용/통합 데이터에 각각 그대로 적용해 '데이터 효과'부터 본다
per_inner, _, _, _, n_per = train_mlp_cfg(G3, EXP017_MLP, SEARCH_SEEDS, unified=False, need_full=False)
base_inner, _, _, _, n_uni = train_mlp_cfg(G3, EXP017_MLP, SEARCH_SEEDS, unified=True, need_full=False)
SC_PER, SC_UNI = g3_inner_score(per_inner), g3_inner_score(base_inner)
print(f"  [대조] exp017 설정 + 전용 데이터({n_per:,}행): g3 내부검증 {SC_PER:.4f}")
print(f"  [기준] exp017 설정 + 통합 데이터({n_uni:,}행): g3 내부검증 {SC_UNI:.4f}   "
      f"<- 데이터 효과 {SC_UNI - SC_PER:+.4f}")


def try_cfg(**over):
    """exp017 설정에서 일부만 바꿔 학습하고 g3 내부검증 점수를 낸다."""
    cfg = {**EXP017_MLP, **over}
    inner, _, _, _, _ = train_mlp_cfg(G3, cfg, SEARCH_SEEDS, unified=True, need_full=False)
    return g3_inner_score(inner), cfg


# ---------------- 1단계: 학습률 ----------------
# full-batch라 배치 크기 = 데이터 크기다. 5.5배가 됐으므로 exp017 값(1e-3) 위쪽까지 연다.
LR_GRID = [2e-4, 3e-4, 5e-4, 1e-3, 2e-3, 3e-3, 5e-3]
print(f"\n--- 1단계: 학습률 격자 (구조는 exp017 {EXP017_MLP['hidden']} 고정) ---")
lr_scores = {}
for lr in LR_GRID:
    lr_scores[lr], _ = try_cfg(lr=lr)
    mark = "  <- exp017 값" if lr == EXP017_MLP["lr"] else ""
    print(f"    lr={lr:.1e}  {lr_scores[lr]:.4f}{mark}")
BEST_LR = max(lr_scores, key=lr_scores.get)
print(f"  -> 최적 lr={BEST_LR:.1e} ({lr_scores[BEST_LR]:.4f}, exp017 대비 {lr_scores[BEST_LR]-SC_UNI:+.4f})")
if BEST_LR in (LR_GRID[0], LR_GRID[-1]):
    print(f"  ⚠ 격자 경계에서 멈췄다 — 그 방향으로 더 열어야 할 수 있다")

# ---------------- 2단계: 은닉층 구조 ----------------
ARCH_CHOICES = [(256, 256),            # 0: exp017 구조
                (128, 128),            # 1
                (384, 384),            # 2
                (512, 512),            # 3
                (256, 256, 256),       # 4
                (384, 384, 384),       # 5
                (512, 256),            # 6: 좁아지는 형태
                (384, 192),            # 7
                (512, 256, 128)]       # 8
print(f"\n--- 2단계: 은닉층 구조 전수 (lr={BEST_LR:.1e} 고정) ---")
arch_scores = {}
for a in ARCH_CHOICES:
    arch_scores[a], _ = try_cfg(lr=BEST_LR, hidden=a, n_layers=len(a))
    print(f"    {str(a):<18} {arch_scores[a]:.4f}")
BEST_ARCH = max(arch_scores, key=arch_scores.get)
print(f"  -> 최적 구조 {BEST_ARCH} ({arch_scores[BEST_ARCH]:.4f}, "
      f"exp017 구조 대비 {arch_scores[BEST_ARCH]-arch_scores[(256,256)]:+.4f})")

# ---------------- 3단계: 미세조정 (구조가 바뀌었으니 lr을 다시 본다) ----------------
print(f"\n--- 3단계: {BEST_ARCH} 위에서 lr·p_drop·weight_decay·t_soft 동시 미세조정 ---")


def mlp_objective(tr):
    cfg = dict(hidden=BEST_ARCH, n_layers=len(BEST_ARCH),
               lr=tr.suggest_float("lr", BEST_LR / 3, BEST_LR * 3, log=True),
               p_drop=tr.suggest_float("p_drop", 0.0, 0.30),
               weight_decay=tr.suggest_float("weight_decay", 1e-5, 1e-3, log=True),
               t_soft=tr.suggest_categorical("t_soft", [0.003, 0.006, 0.012]))
    inner, _, _, _, _ = train_mlp_cfg(G3, cfg, SEARCH_SEEDS, unified=True, need_full=False)
    sc = g3_inner_score(inner)
    print(f"    시행 {tr.number:>2}: lr={cfg['lr']:.2e} drop={cfg['p_drop']:.3f} "
          f"wd={cfg['weight_decay']:.2e} T={cfg['t_soft']} -> {sc:.4f}")
    return sc


# n_startup_trials: TPE가 무작위로만 뽑는 초기 시행 수. 기본 10이면 15회 중 5회만
# 모델 안내를 받는다. 첫 시행에 exp017 값을 넣어 좋은 점을 하나 알고 시작하므로 5로 낮춘다.
sm = optuna.create_study(direction="maximize",
                         sampler=optuna.samplers.TPESampler(seed=RNG_SEED, n_startup_trials=5))
sm.enqueue_trial({"lr": BEST_LR, "p_drop": EXP017_MLP["p_drop"],          # 2단계 결과에서 출발
                  "weight_decay": EXP017_MLP["weight_decay"], "t_soft": EXP017_MLP["t_soft"]})
sm.optimize(mlp_objective, n_trials=N_TRIALS_MLP, show_progress_bar=False)

MLP_UNI_G3 = dict(hidden=BEST_ARCH, n_layers=len(BEST_ARCH), **sm.best_params)

print(f"\n{'='*74}")
print(f"  [최종] 은닉층 {MLP_UNI_G3['hidden']} | lr={MLP_UNI_G3['lr']:.2e} "
      f"drop={MLP_UNI_G3['p_drop']:.3f} wd={MLP_UNI_G3['weight_decay']:.2e} "
      f"T_soft={MLP_UNI_G3['t_soft']}")
print(f"  [exp017] 은닉층 (256, 256) | lr=1.00e-03 drop=0.150 wd=1.00e-04 T_soft=0.006")
print(f"{'='*74}")
print(f"  g3 내부검증 단계별: 전용 {SC_PER:.4f} -> 통합 {SC_UNI:.4f} "
      f"-> +lr {lr_scores[BEST_LR]:.4f} -> +구조 {arch_scores[BEST_ARCH]:.4f} "
      f"-> +미세조정 {sm.best_value:.4f}")
print(f"  소요 {time.time()-t0:.0f}초")

**실행 후 확인할 것**

- **단계별 분해 한 줄**이 이 절의 요약이다: `전용 → 통합(데이터 효과) → +lr → +구조 → +미세조정`. 어느 단계가 실제로 값을 했는지 여기서 갈린다.
- **1단계**: 최적 lr이 exp017 값(`1e-3`)에서 얼마나 움직였는지. **격자 경계(`2e-4` 또는 `5e-3`)에서 멈추면 경고가 뜬다** — 그러면 그 방향으로 더 열어야 한다.
- **2단계**: 최적 구조가 `(256,256)`보다 나은지, 그리고 **넓어지는 쪽인지 깊어지는 쪽인지 좁아지는 쪽인지**.
- **3단계**: 구조가 바뀐 뒤 lr이 1단계 값에서 또 움직였는지. 많이 움직였다면 lr과 구조의 상호작용이 실제로 컸다는 뜻이다.

### 5-3. 블렌드 `w` — 칸마다 따로

g3의 두 모델 중 하나라도 바뀌었으므로 `w`를 다시 고른다. **g1·g2는 exp017 값 유지**(아무것도 안 바뀌었다). 추가 학습 없이 내부검증 예측만 재사용한다.

In [ ]:
LGB_EXP017 = {g: {**LGB_BASE, **EXP017_LGB_CORE, "alpha": EXP017_TAU[g]} for g in TARGET_COLS}

LGB_OUT = {}
for g in TARGET_COLS:
    LGB_OUT[("exp017", g)] = train_lgb_group(g, LGB_EXP017[g])
LGB_OUT[("uni", G3)] = train_lgb_unified_g3(LGB_UNI_G3)
print("LightGBM 학습 완료: " + " ".join(f"{k[1][-1]}/{k[0]}:{v[2]['n_trees']}그루"
                                      for k, v in LGB_OUT.items()))

MLP_OUT = {}
for g in ["kpx_group_1", "kpx_group_2"]:
    MLP_OUT[("exp017", g)] = train_mlp_cfg(g, EXP017_MLP, NN_SEEDS, unified=False)
MLP_OUT[("exp017", G3)] = train_mlp_cfg(G3, EXP017_MLP, NN_SEEDS, unified=False)
MLP_OUT[("uni", G3)] = train_mlp_cfg(G3, MLP_UNI_G3, NN_SEEDS, unified=True)
print("MLP 학습 완료: " + " ".join(f"{k[1][-1]}/{k[0]}:{v[1]}에폭({v[4]:,}행)" for k, v in MLP_OUT.items()))


def pick_w(g, lgb_key, mlp_key):
    m = MASKS[g]; cap = CAPACITY_KWH[g]
    act = feat.loc[m["inner_va"], g].to_numpy()
    pl, pm = LGB_OUT[(lgb_key, g)][1], MLP_OUT[(mlp_key, g)][0]
    s, w = max((group_score(act, np.clip((1 - x) * pl + x * pm, 0, cap), cap)[0], x) for x in W_GRID)
    return float(w), s


w_B, sB = pick_w(G3, "exp017", "uni")
w_C, sC = pick_w(G3, "uni", "uni")
w_A, sA = pick_w(G3, "exp017", "exp017")
print(f"\ng3 블렌드 w — exp017 {EXP017_W[G3]:.2f} (같은 절차로 다시 뽑으면 {w_A:.2f}, 내부검증 {sA:.4f})")
print(f"  칸 B (전용LGB + 통합MLP): w={w_B:.2f} (내부검증 {sB:.4f})")
print(f"  칸 C (통합LGB + 통합MLP): w={w_C:.2f} (내부검증 {sC:.4f})")

**실행 후 확인할 것**

- **B의 `w`가 exp017 값 0.90보다 올랐는지** — 통합으로 MLP가 좋아졌다면 MLP 비중이 커지는 게 자연스럽다.
- **C의 `w`는 반대로 내려갈 수 있다** — LightGBM도 좋아졌다면 다시 섞을 가치가 생기기 때문이다. `06` §12에서 `w(g3)=1.00`(LightGBM 완전 배제)이 나왔던 것과 대비해 보면 의미가 있다.

## 6. 2024 홀드아웃 채점 — 세 칸

세 칸 모두 **g1·g2는 exp017 그대로**이고 g3만 다르다. MLP·LightGBM은 §5-3에서 이미 학습했으므로 여기서는 조합만 한다(추가 학습 없음).

In [ ]:
def score(name, lgb_key_g3, mlp_key_g3, w3):
    p_lgb, p_mlp, p_bl, p_seeds, info = {}, {}, {}, {}, {}
    for g in TARGET_COLS:
        cap = CAPACITY_KWH[g]
        lk = lgb_key_g3 if g == G3 else "exp017"
        mk = mlp_key_g3 if g == G3 else "exp017"
        w = w3 if g == G3 else EXP017_W[g]
        p_lgb[g] = LGB_OUT[(lk, g)][0]
        p_mlp[g], p_seeds[g] = MLP_OUT[(mk, g)][2], MLP_OUT[(mk, g)][3]
        p_bl[g] = np.clip((1 - w) * p_lgb[g] + w * p_mlp[g], 0, cap)
        info[g] = {"lgb": LGB_OUT[(lk, g)][2]["mode"], "n_trees": LGB_OUT[(lk, g)][2]["n_trees"],
                   "mlp": "unified" if mk == "uni" else "per-group",
                   "n_epochs": MLP_OUT[(mk, g)][1], "mlp_rows": MLP_OUT[(mk, g)][4], "w": w}
    out = {"name": name, "w3": w3, "info": info, "pred": {"lgb": p_lgb, "mlp": p_mlp, "blend": p_bl},
           "pred_seeds": p_seeds, "w": {g: (w3 if g == G3 else EXP017_W[g]) for g in TARGET_COLS}}
    for k, pr in [("lgb", p_lgb), ("mlp", p_mlp), ("blend", p_bl)]:
        t, n, f = metric(HOLDOUT_ANS, pd.DataFrame(pr))
        out[k] = {"total": t, "one_minus_nmae": n, "ficr": f}
    out["per_group"] = {g: group_score(HOLDOUT_ANS[g], p_bl[g], CAPACITY_KWH[g])[0] for g in TARGET_COLS}
    return out


runs = {"A": score("A 확정 모델(exp017)", "exp017", "exp017", EXP017_W[G3]),
        "B": score("B g3 MLP통합",      "exp017", "uni", w_B),
        "C": score("C g3 MLP+LGB통합",  "uni", "uni", w_C)}
for k, r in runs.items():
    i3 = r["info"][G3]
    print(f"{r['name']:<20} 블렌드 {r['blend']['total']:.4f} | "
          f"g3: LGB {i3['lgb']}({i3['n_trees']}그루) MLP {i3['mlp']}({i3['mlp_rows']:,}행) w={i3['w']:.2f}")

ref = 0.6458
got = runs["A"]["blend"]["total"]
print(f"\n[하네스 이식 검증] 칸 A {got:.4f} vs reports/train.md 기록 {ref:.4f} (차이 {got-ref:+.4f})")
print("  ->", "✔ 재현됨. 아래 비교를 신뢰할 수 있다." if abs(got - ref) < 5e-4
      else "★ 재현 실패 — 이식 사고다. 아래를 보지 말 것.")

**실행 후 확인할 것**

- **이식 검증이 ✔ 여야 한다.** 칸 A가 `0.6458`을 넷째 자리까지 재현해야 06에서 옮겨온 코드가 온전한 것이다.
- 칸 B의 g3: `LGB per-group` + `MLP unified(26,686행)`. 칸 C: `LGB unified` + `MLP unified`.

## 7. 판정

| 비교 | 묻는 것 |
|---|---|
| **B − A**, **C − A** | **확정 모델을 넘었는가** ← 제출 판단 |
| **C − B** | 통합이 **LightGBM에도** 듣는가 |

노이즈는 시드 5개 각각으로 블렌드를 만들어 재고, 비교마다 **그 비교에 참여하는 두 칸만의** 쌍별 표준오차 `sqrt(s_a²+s_b²)`를 쓴다.

**⚠️ 이 배수는 채택 판정이 아니다.** `06` §13에서 홀드아웃 이득이 리더보드 이득을 예측하지 못하는 것이 확인됐다. **제출할 가치가 있는지**를 정하는 데만 쓴다.

In [ ]:
def blend_from_seed(r, si):
    pr = {}
    for g in TARGET_COLS:
        cap = CAPACITY_KWH[g]; w = r["w"][g]
        pr[g] = np.clip((1 - w) * r["pred"]["lgb"][g] + w * r["pred_seeds"][g][si], 0, cap)
    return metric(HOLDOUT_ANS, pd.DataFrame(pr))[0]


rows, noise = [], {}
for k, r in runs.items():
    st_ = np.array([blend_from_seed(r, i) for i in range(len(NN_SEEDS))])
    noise[k] = st_.std(ddof=1) / np.sqrt(len(NN_SEEDS))
    rows.append({"칸": r["name"], "w(g3)": r["w3"], "LGB": round(r["lgb"]["total"], 4),
                 "MLP": round(r["mlp"]["total"], 4), "블렌드": round(r["blend"]["total"], 4),
                 "1-NMAE": round(r["blend"]["one_minus_nmae"], 4),
                 "FICR": round(r["blend"]["ficr"], 4), "시드노이즈": round(noise[k], 5)})
print(pd.DataFrame(rows).to_string(index=False))

TOT = {k: r["blend"]["total"] for k, r in runs.items()}
print(f"\n{'='*80}")
for a, b, note in [("B", "A", "★ B가 확정 모델을 넘었는가"), ("C", "A", "★ C가 확정 모델을 넘었는가"),
                   ("C", "B", "LightGBM까지 통합한 추가 효과")]:
    d = TOT[a] - TOT[b]; se = float(np.hypot(noise[a], noise[b]))
    print(f"  {a} - {b} = {d:+.4f}   (SE {se:.5f}, {d/se if se>0 else float('nan'):+5.1f}배)   {note}")
print(f"  [참고] 06 §5의 A′(완전분리+재튜닝) = 0.6487 — 같은 하네스·같은 시드의 결정론적 결과")
print(f"{'='*80}")

print("\n--- 그룹별 블렌드 (g3만 달라져야 한다) ---")
print(pd.DataFrame({r["name"]: {g: round(r["per_group"][g], 4) for g in TARGET_COLS}
                    for r in runs.values()}).T.to_string())
print("\n--- 단독 점수 ---")
for k, r in runs.items():
    print(f"  {r['name']:<20} LGB {r['lgb']['total']:.4f} | MLP {r['mlp']['total']:.4f}")

best = max(["B", "C"], key=lambda k: TOT[k])
d = TOT[best] - TOT["A"]; g3d = runs[best]["per_group"][G3] - runs["A"]["per_group"][G3]
print(f"\n{'='*80}")
if d > 0 and g3d > 0:
    print(f"[판정] 최고 후보 {best}. 확정 모델을 {d:+.4f} 넘었고 g3도 {g3d:+.4f} 올랐다.")
    print(f"       -> train.ipynb에 {best} 구성을 반영해 **제출**한다. 리더보드가 최종 판정한다.")
elif d > 0:
    print(f"[판정] {best}의 총점은 {d:+.4f}인데 g3는 {g3d:+.4f}다. 기전이 안 맞는다 — 원인부터 볼 것.")
else:
    print(f"[판정] B·C 모두 확정 모델을 넘지 못했다(최고 {best} {d:+.4f}, g3 {g3d:+.4f}).")
    print("       -> 근거 6개짜리 후보가 이 파이프라인에서는 안 통한다. 기록하고 다음 후보로.")
print(f"{'='*80}")

**실행 후 확인할 것**

- **`B − A` / `C − A`가 결론**이다. 그리고 **그룹별 표에서 g3만 움직였는지** 확인한다 — g1·g2는 세 칸이 완전히 같은 모델이라 **소수점 넷째 자리까지 같아야 한다.** 다르면 코드에 사고가 있다.
- **`C − B`** 가 통합이 LightGBM에도 듣는지를 가른다. 양수면 표본 부족이 두 모델 다의 문제였다는 뜻이다.

## 8. 실험 로그 기록

In [ ]:
def git_state():
    try:
        h = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
        d = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout.strip()
        return f"{h}({'dirty' if d else 'clean'})"
    except Exception:
        return "unknown"


def by_group(ans, pred):
    out = {}
    for g in TARGET_COLS:
        a = ans[g].to_numpy(float); f_ = pred[g].to_numpy(float); cap = CAPACITY_KWH[g]
        v = a >= cap * SCORE_THRESHOLD; a, f_ = a[v], f_[v]
        er = np.abs(f_ - a) / cap
        pr = np.select([er <= 0.06, er <= 0.08], [4.0, 3.0], default=0.0)
        out[g] = {"nmae": float(er.mean()), "ficr": float((a * pr).sum() / (a * 4.0).sum())}
    return out


LOG_PATH = EXP_DIR / "log.csv"
log = pd.read_csv(LOG_PATH, encoding="utf-8-sig")
next_id = max(int(s[3:]) for s in log["exp_id"] if str(s).startswith("exp")) + 1

new_rows = []
for k in ["B", "C"]:                       # A는 exp017과 같은 구성이라 기록하지 않는다
    r = runs[k]; bg = by_group(HOLDOUT_ANS, pd.DataFrame(r["pred"]["blend"]))
    row = {"exp_id": f"exp{next_id:03d}", "date": pd.Timestamp.now().strftime("%Y-%m-%d"),
           "git_hash": git_state(),
           "model": f"blend(lgbm+mlp) g3={'MLP+LGB' if k == 'C' else 'MLP'}통합+재튜닝",
           "features": f"07_{k}_g3unified", "total_score": round(r["blend"]["total"], 6),
           "one_minus_nmae": round(r["blend"]["one_minus_nmae"], 6),
           "ficr": round(r["blend"]["ficr"], 6),
           "val_period": "2024-01-01~2024-12-31", "fit_seconds": "", "public_score": "",
           "note": (f"07 {r['name']} | g3만 재튜닝(g1·g2 확정 모델 유지) | MLP {MLP_UNI_G3} | "
                    f"w(g3)={r['w3']:.2f} | vs A {r['blend']['total']-runs['A']['blend']['total']:+.4f} | "
                    f"LGB단독 {r['lgb']['total']:.4f} MLP단독 {r['mlp']['total']:.4f}")}
    for g in TARGET_COLS:
        row[f"nmae_g{g[-1]}"] = round(bg[g]["nmae"], 6)
        row[f"ficr_g{g[-1]}"] = round(bg[g]["ficr"], 6)
    new_rows.append(row); next_id += 1

new_df = pd.DataFrame(new_rows)
assert not [c for c in new_df.columns if c not in log.columns]
pd.concat([log, new_df], ignore_index=True)[log.columns].to_csv(LOG_PATH, index=False, encoding="utf-8-sig")
print(f"{len(new_rows)}개 행 추가")
print(new_df[["exp_id", "features", "total_score", "one_minus_nmae", "ficr"]].to_string(index=False))

**실행 후 확인할 것**

- **이 셀은 여러 번 돌리면 행이 중복으로 쌓인다. 한 번만 실행한다.**

## 9. 종합 해석

*(실행 후 채운다.)*

| 칸 | g3 LGB | g3 MLP | w(g3) | 블렌드 | 1−NMAE | FICR | g3 점수 |
|---|---|---|---|---|---|---|---|
| A 확정 모델 | 전용 | 전용 256/2/0.15 | 0.90 | | | | |
| B | 전용 | 통합 (튜닝값) | | | | | |
| C | 통합 | 통합 (튜닝값) | | | | | |

**g3 MLP 튜닝 결과**: 은닉층 `(?, ?)` · `p_drop=?` · `lr=?` · `weight_decay=?` · `t_soft=?`
(exp017은 `(256, 256)` · `0.15` · `1e-3` · `1e-4` · `0.006`)

**세 단계 분해** (내부검증 g3): exp017설정+전용 `?` → exp017설정+통합 `?`(데이터 효과) → 튜닝완료+통합 `?`(재튜닝 효과)

### 다음 행동

- **B나 C가 A를 넘고 g3도 올랐다면** → 그 구성을 `train.ipynb`에 반영하고 **제출**한다.
  - `CONFIGS`에 `g3_unified` 스위치와 **그룹별 MLP 아키텍처**를 넣고, `config.json`에 기록한다.
  - ⚠️ **통합 모델은 입력이 182차원**(원-핫 3개)이고 **구조도 다르다**(`hidden`/`n_layers`/`p_drop`). `inference.ipynb`가 `MLP(len(feature_cols))`로 만들면 **차원도 구조도 안 맞아 실패**한다. `config.json`의 아키텍처를 읽어 재구성하도록 고쳐야 한다.
  - ⚠️ C를 채택하면 **g3 LightGBM 타깃이 이용률**이므로 예측 후 설비용량을 곱하는 것도 반영해야 한다.
- **둘 다 못 넘었다면** → 근거 6개짜리 후보가 이 파이프라인에서 안 통한다는 뜻이다. 기록하고 다음 후보로 넘어간다.